# The Training Loop: Model Engine Room

Reach for this when you need: 
- Standard boilerplate for training, validation, and early stopping.
- Reference for saving and loading full model checkpoints.
- Logic for switching between `model.train()` and `model.eval()`.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Standard Boilerplate Loop

| State | Description | Purpose |
| :--- | :--- | :--- |
| `model.train()` | Enables Dropout and BatchNorm update | Ensures weights learn |
| `model.eval()` | Disables Dropout, freezes BatchNorm | Consistent predictions for validation |
| `torch.no_grad()` | Disables gradient graph construction | Memory and compute optimization during inference |

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for x, y in tqdm(loader):
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()
        outputs = model(x)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    return total_loss / len(loader)

def validate(model, loader, criterion):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            loss = criterion(outputs, y)
            total_loss += loss.item()
    return total_loss / len(loader)

## 2. Checkpointing: Saving & Loading

**Always save the state_dict, not the model object.**

✅ **Use when**: Capturing training progress every N epochs or when val loss hits a new minimum.
❌ **Don't use when**: Deploying for inference where you might want to switch to TorchScript or ONNX (though `state_dict` still works for both).

In [ ]:
checkpoint = {
    'epoch': 10,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': 0.05,
}

torch.save(checkpoint, "best_model.pth")

# Loading
checkpoint = torch.load("best_model.pth")
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

### Common Pitfalls
- **Evaluation mode**: If you forget `model.eval()`, Dropout stays active, making validation metrics unstable.
- **Item loss**: Always call `loss.item()` when tracking metrics. Storing raw `loss` (the tensor) results in a memory leak because the CPU keeps the entire graph alive.
- **Optimizer state**: If you resume training from a checkpoint, you MUST load the `optimizer_state_dict` as well to restore momentum/running averages.

### Key Takeaways
- Separate training and validation into clearly defined functions for modularity.
- Checkpointing is essential for long-running jobs to handle random failures.
- `tqdm` provides vital visibility into progress for scannable training feedback.